Objectives:

1. Configure a PySpark execution environment suitable for large-scale data processing.

2. Ingest a multi-gigabyte dataset stored in Parquet format.

3. Validate data schema and assess initial data quality at ingestion.

4. Apply partitioning strategies aligned with distributed query execution.

5. Demonstrate memory-aware processing through selective caching and unpersisting.

6. Justify the use of the DataFrame API over RDDs for scalable data engineering.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC-Taxi-BigData-Classification") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark

In [4]:
import os

data_path = "/content/drive/MyDrive/bigdata_ml_project/data/raw/nyc_taxi"

total_size = 0
for dirpath, dirnames, filenames in os.walk(data_path):
    for f in filenames:
        fp = os.path.join(dirpath, f)
        total_size += os.path.getsize(fp)

total_size_gb = total_size / (1024 ** 3)
total_size_gb

1.349741299636662

In [5]:
data_path = "/content/drive/MyDrive/bigdata_ml_project/data/raw/nyc_taxi"

df = spark.read.parquet(data_path)

df.printSchema()
df.count()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



85587316

In [6]:
# Number of columns
len(df.columns)

# Basic null check
from pyspark.sql.functions import col

null_counts = df.select([
    col(c).isNull().cast("int").alias(c)
    for c in df.columns
])

null_counts.groupBy().sum().show(truncate=False)

+-------------+-------------------------+--------------------------+--------------------+------------------+---------------+-----------------------+-----------------+-----------------+-----------------+----------------+----------+------------+---------------+-----------------+--------------------------+-----------------+-------------------------+----------------+
|sum(VendorID)|sum(tpep_pickup_datetime)|sum(tpep_dropoff_datetime)|sum(passenger_count)|sum(trip_distance)|sum(RatecodeID)|sum(store_and_fwd_flag)|sum(PULocationID)|sum(DOLocationID)|sum(payment_type)|sum(fare_amount)|sum(extra)|sum(mta_tax)|sum(tip_amount)|sum(tolls_amount)|sum(improvement_surcharge)|sum(total_amount)|sum(congestion_surcharge)|sum(Airport_fee)|
+-------------+-------------------------+--------------------------+--------------------+------------------+---------------+-----------------------+-----------------+-----------------+-----------------+----------------+----------+------------+---------------+---------

In [7]:
selected_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount"
]

df_small = df.select(selected_cols)

In [8]:
# Logical & physical plan
df_small.explain(True)

== Parsed Logical Plan ==
'Project ['tpep_pickup_datetime, 'tpep_dropoff_datetime, 'PULocationID, 'DOLocationID, 'passenger_count, 'trip_distance, 'fare_amount, 'total_amount]
+- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] parquet

== Analyzed Logical Plan ==
tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, PULocationID: int, DOLocationID: int, passenger_count: bigint, trip_distance: double, fare_amount: double, total_amount: double
Project [tpep_pickup_datetime#1, tpep_dropoff_datetime#2, PULocationID#7, DOLocationID#8, passenger_count#3L, trip_distance#4, fare_amount#10, total_amount#16]
+- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,t

In [9]:
# Partition count
df_small.rdd.getNumPartitions()

16

In [10]:
df_small = df_small.repartition(200)

In [11]:
# Cache confirmation
df_small.cache()
df_small.count()
df_small.is_cached

True

In [12]:
df_small.unpersist()

DataFrame[tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, PULocationID: int, DOLocationID: int, passenger_count: bigint, trip_distance: double, fare_amount: double, total_amount: double]

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(4040, height=600)